In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spikeinterface as si
from spikeinterface.preprocessing import notch_filter
from mne.time_frequency import psd_array_multitaper
import sys 

REPO_ROOT = Path("..").resolve()
sys.path.append(str(REPO_ROOT))

EPHYS_DIR = REPO_ROOT / "DATA" / "ephys"
DATA_DIR = REPO_ROOT / "DATA"

ANALYSIS_WINDOWS_FILE = DATA_DIR / "analysis_windows.csv"

NOTCH_FREQUENCY_HZ = 50
NOTCH_Q = 35

FREQUENCIES_HZ = np.linspace(1, 100, 100)
FMIN = float(FREQUENCIES_HZ.min())
FMAX = float(FREQUENCIES_HZ.max())

OUTPUT_FILE = DATA_DIR / "lfp_power.csv"

In [ ]:
# Load and validate the synchronized analysis windows used for LFP analysis

analysis_windows = pd.read_csv(
    ANALYSIS_WINDOWS_FILE
)

print(
    f"Loaded {len(analysis_windows):,} analysis windows."
)

print(
    f"Recordings: "
    f"{analysis_windows['recording'].nunique()}"
)

print(
    f"Conditions: "
    f"{sorted(analysis_windows['label'].unique())}"
)

required_columns = {
    "recording",
    "segment_index",
    "label",
    "sample_start",
    "sample_end",
}

missing_columns = (
    required_columns
    - set(analysis_windows.columns)
)

if missing_columns:
    raise ValueError(
        f"analysis_windows.csv is missing: "
        f"{sorted(missing_columns)}"
    )

In [ ]:
def calculate_multitaper_power(
    traces: np.ndarray,
    sampling_frequency: float,
) -> np.ndarray:
    """Compute multitaper power spectra for each channel."""

    data = traces.T

    psd, frequencies = psd_array_multitaper(
        data,
        sfreq=sampling_frequency,
        fmin=FMIN,
        fmax=FMAX,
        adaptive=True,
        normalization="full",
        verbose=False,
    )

    return np.vstack(
        [
            np.interp(
                FREQUENCIES_HZ,
                frequencies,
                channel_psd,
            )
            for channel_psd in psd
        ]
    )

In [ ]:
# Identify recordings containing the processed LFP data

recording_dirs = sorted(
    {
        path.parent.parent
        for path in EPHYS_DIR.rglob("lfp/meta.json")
    }
)

print(f"Found {len(recording_dirs)} recordings.")

In [ ]:
# Compute regional LFP power for each behavioral condition and recording

power_results = []

for recording_dir in recording_dirs:

    recording_name = recording_dir.name

    print(f"\nProcessing: {recording_name}")

    lfp_path = recording_dir / "lfp"

    recording = si.load(lfp_path)

    sampling_frequency = float(
        recording.get_sampling_frequency()
    )

    with (lfp_path / "meta.json").open("r") as file:
        metadata = json.load(file)

    pfc_channels = metadata["anatomical_assignment"]["pfc_channels"]
    rsc_channels = metadata["anatomical_assignment"]["rsc_channels"]

    traces = recording.get_traces()

    lfp_data = pd.DataFrame(
        traces,
        columns=recording.channel_ids,
    )

    regional_lfp = pd.DataFrame(
        {
            "PFC": lfp_data[pfc_channels].mean(axis=1),
            "RSC": lfp_data[rsc_channels].mean(axis=1),
        }
    )

    regional_recording = si.NumpyRecording(
        regional_lfp.values,
        sampling_frequency=sampling_frequency,
        channel_ids=["PFC", "RSC"],
    )

    filtered_recording = notch_filter(
        regional_recording,
        freq=NOTCH_FREQUENCY_HZ,
        q=NOTCH_Q,
    )

    recording_windows = analysis_windows[
        analysis_windows["recording"] == recording_name
    ]

    for label, condition_windows in recording_windows.groupby(
        "label"
    ):

        power_stack = []

        for _, window in condition_windows.iterrows():

            sample_start = int(window["sample_start"])
            sample_end = int(window["sample_end"])

            if sample_end <= sample_start:
                continue

            lfp_segment = filtered_recording.get_traces(
                start_frame=sample_start,
                end_frame=sample_end,
                channel_ids=["PFC", "RSC"],
            )

            power = calculate_multitaper_power(
                lfp_segment,
                sampling_frequency,
            )

            power_stack.append(power)

        if not power_stack:
            continue

        mean_power = np.mean(
            power_stack,
            axis=0,
        )

        for channel_index, region in enumerate(
            ["PFC", "RSC"]
        ):

            for frequency_index, frequency in enumerate(
                FREQUENCIES_HZ
            ):

                power_results.append(
                    {
                        "recording": recording_name,
                        "label": label,
                        "region": region,
                        "frequency_hz": frequency,
                        "abs_power": mean_power[
                            channel_index,
                            frequency_index,
                        ],
                    }
                )

In [ ]:
# Assemble and save the LFP power results

power_df = pd.DataFrame(power_results)

print(f"Generated {len(power_df):,} rows.")

power_df.to_csv(
    OUTPUT_FILE,
    index=False,
)

print(f"Saved: {OUTPUT_FILE}")

In [ ]:
# Reshape power spectra by condition and calculate the L1/L2 power ratio

ratio_df = (
    power_df
    .pivot_table(
        index=["recording", "region", "frequency_hz"],
        columns="label",
        values="abs_power",
        aggfunc="first",
    )
    .reset_index()
)

required_labels = {1, 2}

missing_labels = required_labels - set(ratio_df.columns)

if missing_labels:
    raise ValueError(
        f"Cannot calculate L1/L2 ratio. "
        f"Missing labels: {sorted(missing_labels)}"
    )

ratio_df["power_ratio"] = (
    ratio_df[1] / ratio_df[2]
)

ratio_df = ratio_df[
    [
        "recording",
        "region",
        "frequency_hz",
        "power_ratio",
    ]
]

In [ ]:
# Save the power-ratio dataset and summarize ratios across recordings

RATIO_OUTPUT_FILE = DATA_DIR / "lfp_power_ratio.csv"

ratio_df.to_csv(
    RATIO_OUTPUT_FILE,
    index=False,
)

print(f"Saved ratio dataset to: {RATIO_OUTPUT_FILE}")

ratio_summary = (
    ratio_df
    .groupby(
        ["region", "frequency_hz"],
        as_index=False,
    )
    .agg(
        mean_ratio=("power_ratio", "mean"),
        sem_ratio=(
            "power_ratio",
            lambda values: (
                values.std(ddof=1) / np.sqrt(len(values))
                if len(values) > 1
                else 0.0
            ),
        ),
        n=("power_ratio", "size"),
    )
)